<a href="https://colab.research.google.com/github/marcosptz/tcc-sistemas-informacao/blob/main/TCC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade ultralytics
!pip install -q gradio opencv-python ultralytics

import os
import shutil
import sys
import cv2
import gradio as gr
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# 1. Atualiza o repositório GitHub
if os.path.exists('/content/projeto_tcc'):
  shutil.rmtree('/content/projeto_tcc')

!git clone https://github.com/marcosptz/tcc-sistemas-informacao.git /content/projeto_tcc

if '/content/projeto_tcc' not in sys.path:
  sys.path.append('/content/projeto_tcc')

from logic import TrackerComportamental

# Modelos oficiais pré treinados na COCO
model = YOLO('yolo11s.pt')
model1 = YOLO('yolov8n.pt')
model2 = YOLO('yolov8s.pt')
model3 = YOLO('yolov8m.pt')
model4 = YOLO('yolov8l.pt')
model5 = YOLO('yolov8x.pt')
model6 = YOLO('/content/drive/MyDrive/TCC_Resultados/treino_lixo_v1/weights/best.pt')

# 2. Inicializa o Tracker com o modelo COCO (yolov8n.pt)
tracker = TrackerComportamental(
    model_path=model, limite_tempo_estatico_segundos=3
)

# 3. DEFINE A FUNÇÃO PROCESSAR_VIDEO_COLAB
def processar_video_colab(video_path):
  if video_path is None:
    return None

  cap = cv2.VideoCapture(video_path)
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30

  temp_output = '/content/temp_processado.mp4'
  final_output = '/content/video_processado.mp4'

  fourcc = cv2.VideoWriter_fourcc(*'mp4v')
  out = cv2.VideoWriter(temp_output, fourcc, fps, (width, height))

  while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
      break

    # Processa o frame com a lógica de detecção e tempo estático
    frame_anotado, _ = tracker.processar_frame(frame)
    out.write(frame_anotado)

  cap.release()
  out.release()

  # Converte para H.264 usando FFmpeg para compatibilidade com o navegador no Gradio
  os.system(f'ffmpeg -y -i {temp_output} -vcodec libx264 {final_output}')

  return final_output


# 4. Inicializa e lança a Interface do Gradio
demo = gr.Interface(
    fn=processar_video_colab,
    inputs=gr.Video(label="Upload do Vídeo de Teste"),
    outputs=gr.Video(label="Vídeo com Detecção e Alertas"),
    title="Sistema de Monitoramento com IA - COCO Model",
)

demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/projeto_tcc'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 48 (delta 19), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 12.90 KiB | 4.30 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ae99c00aeddf0d19b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 736x1280 1 car, 1 parking meter, 29.5ms
Speed: 35.2ms preprocess, 29.5ms inference, 20.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 2 cars, 1 parking meter, 29.7ms
Speed: 7.9ms preprocess, 29.7ms inference, 1.6ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 2 cars, 1 parking meter, 29.3ms
Speed: 6.2ms preprocess, 29.3ms inference, 1.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 2 cars, 1 parking meter, 29.5ms
Speed: 6.8ms preprocess, 29.5ms inference, 1.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 car, 1 parking meter, 29.4ms
Speed: 6.2ms preprocess, 29.4ms inference, 1.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 2 cars, 1 parking meter, 29.3ms
Speed: 6.4ms preprocess, 29.3ms inference, 1.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 2 cars, 1 parking meter, 29.4ms
Speed: 7.5ms preprocess, 29.4ms inference, 1.9ms postprocess per image at shape (1, 3, 736, 1280)

In [2]:
!pip install -q --upgrade ultralytics gradio opencv-python

import os
import shutil
import sys
import cv2
import gradio as gr
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

# 1. Atualiza o repositório GitHub
if os.path.exists('/content/projeto_tcc'):
  shutil.rmtree('/content/projeto_tcc')

!git clone https://github.com/marcosptz/tcc-sistemas-informacao.git /content/projeto_tcc

if '/content/projeto_tcc' not in sys.path:
  sys.path.append('/content/projeto_tcc')

from logic import TrackerComportamental

# -------------------------------------------------------------
# Escolha qual modelo quer usar (Descomente apenas uma das opções):
# -------------------------------------------------------------

# Opção A: YOLOv8s, YOLO11s, YOLO26s (Nome correto: sem o 'v')
MODEL_PATH = 'yolo26s.pt'

# Opção B: Seu modelo customizado de Lixo treinado no Roboflow
# MODEL_PATH = '/content/drive/MyDrive/TCC_Resultados/treino_lixo_v1/weights/best.pt'

# 2. Inicializa o Tracker passando a STRING com o caminho do modelo
tracker = TrackerComportamental(
    model_path=MODEL_PATH, limite_tempo_estatico_segundos=3
)

# 3. DEFINE A FUNÇÃO PROCESSAR_VIDEO_COLAB
def processar_video_colab(video_path):
  if video_path is None:
    return None

  cap = cv2.VideoCapture(video_path)
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30

  temp_output = '/content/temp_processado.mp4'
  final_output = '/content/video_processado.mp4'

  fourcc = cv2.VideoWriter_fourcc(*'mp4v')
  out = cv2.VideoWriter(temp_output, fourcc, fps, (width, height))

  while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
      break

    # Processa o frame com a lógica de detecção e tempo estático
    frame_anotado, _ = tracker.processar_frame(frame)
    out.write(frame_anotado)

  cap.release()
  out.release()

  # Converte para H.264 usando FFmpeg para compatibilidade com o navegador no Gradio
  os.system(f'ffmpeg -y -i {temp_output} -vcodec libx264 {final_output}')

  return final_output


# 4. Inicializa e lança a Interface do Gradio
demo = gr.Interface(
    fn=processar_video_colab,
    inputs=gr.Video(label="Upload do Vídeo de Teste"),
    outputs=gr.Video(label="Vídeo com Detecção e Alertas"),
    title="Sistema de Monitoramento com IA - COCO Model",
)

demo.launch(share=True, debug=True)

Mounted at /content/drive
Cloning into '/content/projeto_tcc'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 51 (delta 21), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 13.86 KiB | 6.93 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://66e6df7335d1e57f4c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 692ms
Prepared 1 package in 18ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 736x1280 1 car, 945.8ms
Speed: 30.4ms preprocess, 945.8ms inference, 29.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 car, 790.1ms
Speed: 12.2ms preprocess, 790.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 car, 964.2ms
Speed: 10.9ms preprocess, 964.2ms inference, 1.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 car, 960.3ms
Speed: 9.9ms preprocess, 960.3ms inference, 1.7ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 car, 1016.8ms
Speed: 18.4ms preprocess, 1016.8ms inference, 2.4ms postprocess per image at shape (1, 3, 736, 1280)

